# Phase 1: Data Quality, Cleaning & Feature Engineering

## Objective

This notebook establishes a reproducible and defensible preprocessing pipeline for the CDC BRFSS 2015 diabetes datasets.

The workflow:

1. Inspect the available datasets
2. Compare their target distributions
3. Audit missing values, duplicates, data types and invalid values
4. Apply source-compatible validity rules
5. Optimize memory usage
6. Engineer project-defined analytical features
7. Save clean datasets for downstream statistical analysis and modelling

### Primary analytical dataset

The `Diabetes_012` dataset is used as the primary dataset because it preserves three distinct outcome groups:

- `0` = No Diabetes
- `1` = Prediabetes
- `2` = Diabetes

The binary and balanced datasets are retained for validation and machine-learning experiments rather than being treated as separate primary populations.

In [64]:
import os
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add src directory to Python path
sys.path.append("../src")

from preprocessing import (
    audit_dataset,
    clean_invalid_values,
    downcast_types,
    engineer_features
)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries loaded successfully.")

Libraries loaded successfully.


## 1. Dataset Inventory

Three versions of the BRFSS 2015 diabetes dataset are retained:

| Dataset | Target | Purpose |
|---|---|---|
| `diabetes_012_health_indicators_BRFSS2015.csv` | `Diabetes_012` | **Primary analysis** |
| `diabetes_binary_health_indicators_BRFSS2015.csv` | `Diabetes_binary` | Secondary validation |
| `diabetes_binary_5050split_health_indicators_BRFSS2015.csv` | `Diabetes_binary` | Balanced ML benchmark |

The three-class dataset is preferred for the main data story because it allows prediabetes to be examined separately rather than combining it with diabetes.

In [65]:
RAW_DIR = "../data/raw"
PROCESSED_DIR = "../data/processed"

os.makedirs(PROCESSED_DIR, exist_ok=True)

files = {
    "primary": "diabetes_012_health_indicators_BRFSS2015.csv",
    "binary": "diabetes_binary_health_indicators_BRFSS2015.csv",
    "balanced": "diabetes_binary_5050split_health_indicators_BRFSS2015.csv"
}

for name, filename in files.items():
    path = os.path.join(RAW_DIR, filename)

    if os.path.exists(path):
        print(f"✓ {name.title():<10} {filename}")
    else:
        print(f"✗ {name.title():<10} FILE NOT FOUND: {filename}")

✓ Primary    diabetes_012_health_indicators_BRFSS2015.csv
✓ Binary     diabetes_binary_health_indicators_BRFSS2015.csv
✓ Balanced   diabetes_binary_5050split_health_indicators_BRFSS2015.csv


## 2. Load the Raw Datasets

In [66]:
df_primary = pd.read_csv(
    os.path.join(RAW_DIR, files["primary"])
)

df_binary = pd.read_csv(
    os.path.join(RAW_DIR, files["binary"])
)

df_balanced = pd.read_csv(
    os.path.join(RAW_DIR, files["balanced"])
)

print("Primary dataset:", df_primary.shape)
print("Binary dataset:", df_binary.shape)
print("Balanced dataset:", df_balanced.shape)

Primary dataset: (253680, 22)
Binary dataset: (253680, 22)
Balanced dataset: (70692, 22)


In [67]:
dataset_summary = pd.DataFrame({
    "Dataset": [
        "Primary: Diabetes_012",
        "Binary: Full Population",
        "Balanced: 50/50"
    ],
    "Rows": [
        len(df_primary),
        len(df_binary),
        len(df_balanced)
    ],
    "Columns": [
        df_primary.shape[1],
        df_binary.shape[1],
        df_balanced.shape[1]
    ],
    "Target": [
        "Diabetes_012",
        "Diabetes_binary",
        "Diabetes_binary"
    ]
})

display(dataset_summary)

,Dataset,Rows,Columns,Target
0,Primary: Diabetes_012,253680,22,Diabetes_012
1,Binary: Full Population,253680,22,Diabetes_binary
2,Balanced: 50/50,70692,22,Diabetes_binary


## 3. Target Distribution Audit

The target distribution is examined before cleaning so that the class structure of each dataset is documented.

The primary dataset contains three categories:

- `0` — No Diabetes
- `1` — Prediabetes
- `2` — Diabetes

The binary dataset combines prediabetes and diabetes into one category.

The balanced dataset intentionally changes the class proportions and is therefore not used to represent population prevalence.

In [68]:
primary_distribution = (
    df_primary["Diabetes_012"]
    .value_counts()
    .sort_index()
    .to_frame("Count")
)

primary_distribution["Percentage"] = (
    primary_distribution["Count"]
    / len(df_primary)
    * 100
)

primary_distribution.index = [
    "No Diabetes",
    "Prediabetes",
    "Diabetes"
]

display(primary_distribution.round(2))

,Count,Percentage
No Diabetes,213703,84.24
Prediabetes,4631,1.83
Diabetes,35346,13.93


In [69]:
binary_distribution = (
    df_binary["Diabetes_binary"]
    .value_counts()
    .sort_index()
    .to_frame("Count")
)

binary_distribution["Percentage"] = (
    binary_distribution["Count"]
    / len(df_binary)
    * 100
)

binary_distribution.index = [
    "No Diabetes",
    "Prediabetes / Diabetes"
]

display(binary_distribution.round(2))

,Count,Percentage
No Diabetes,218334,86.07
Prediabetes / Diabetes,35346,13.93


In [70]:
balanced_distribution = (
    df_balanced["Diabetes_binary"]
    .value_counts()
    .sort_index()
    .to_frame("Count")
)

balanced_distribution["Percentage"] = (
    balanced_distribution["Count"]
    / len(df_balanced)
    * 100
)

balanced_distribution.index = [
    "No Diabetes",
    "Prediabetes / Diabetes"
]

display(balanced_distribution.round(2))

,Count,Percentage
No Diabetes,35346,50.0
Prediabetes / Diabetes,35346,50.0


## 4. Primary Dataset: Initial Inspection

In [71]:
display(df_primary.head())

,Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,HvyAlcoholConsump,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


In [72]:
print("Shape:", df_primary.shape)

print("\nColumn names:")
for i, column in enumerate(df_primary.columns, start=1):
    print(f"{i:2}. {column}")

Shape: (253680, 22)

Column names:
 1. Diabetes_012
 2. HighBP
 3. HighChol
 4. CholCheck
 5. BMI
 6. Smoker
 7. Stroke
 8. HeartDiseaseorAttack
 9. PhysActivity
10. Fruits
11. Veggies
12. HvyAlcoholConsump
13. AnyHealthcare
14. NoDocbcCost
15. GenHlth
16. MentHlth
17. PhysHlth
18. DiffWalk
19. Sex
20. Age
21. Education
22. Income


In [73]:
df_primary.info()

<class 'pandas.DataFrame'>
RangeIndex: 253680 entries, 0 to 253679
Data columns (total 22 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   Diabetes_012          253680 non-null  float64
 1   HighBP                253680 non-null  float64
 2   HighChol              253680 non-null  float64
 3   CholCheck             253680 non-null  float64
 4   BMI                   253680 non-null  float64
 5   Smoker                253680 non-null  float64
 6   Stroke                253680 non-null  float64
 7   HeartDiseaseorAttack  253680 non-null  float64
 8   PhysActivity          253680 non-null  float64
 9   Fruits                253680 non-null  float64
 10  Veggies               253680 non-null  float64
 11  HvyAlcoholConsump     253680 non-null  float64
 12  AnyHealthcare         253680 non-null  float64
 13  NoDocbcCost           253680 non-null  float64
 14  GenHlth               253680 non-null  float64
 15  MentHlth   

## 5. Comprehensive Data Quality Audit

The audit checks:

- Dataset dimensions
- Missing values
- Duplicate records
- Data types
- Invalid categorical codes
- Invalid ordinal values
- Invalid BMI values

No observations are removed solely because they are duplicated at this stage. Identical survey profiles are reported so that the data structure can be interpreted before preprocessing.

In [74]:
audit_dataset(
    df_primary,
    "Primary Diabetes_012 Dataset — Raw"
)


DATA QUALITY AUDIT: Primary Diabetes_012 Dataset — Raw

Dimensions: 253,680 rows × 22 columns

Total Missing Values: 0



Identical Survey Profiles: 23,899 (9.42%)

Data Types:
float64    22
Name: count, dtype: int64

Invalid Values:
  No invalid coded values detected.


In [75]:
audit_dataset(
    df_balanced,
    "Balanced 50/50 Dataset — Raw"
)


DATA QUALITY AUDIT: Balanced 50/50 Dataset — Raw

Dimensions: 70,692 rows × 22 columns

Total Missing Values: 0

Identical Survey Profiles: 1,635 (2.31%)

Data Types:
float64    22
Name: count, dtype: int64

Invalid Values:
  No invalid coded values detected.


## 6. Missing-Value Profile

In [76]:
missing_profile = (
    df_primary.isnull()
    .sum()
    .sort_values(ascending=False)
    .to_frame("Missing Values")
)

missing_profile["Missing %"] = (
    missing_profile["Missing Values"]
    / len(df_primary)
    * 100
)

display(
    missing_profile[
        missing_profile["Missing Values"] > 0
    ].round(2)
)

,Missing Values,Missing %


## 7. Validate Encoded Values

The BRFSS variables use predefined numerical codes. Before modelling, each variable is checked against its expected valid range.

This prevents accidental treatment of impossible codes as genuine observations.

In [77]:
VALID_RANGES = {
    "Diabetes_012": {0, 1, 2},
    "HighBP": {0, 1},
    "HighChol": {0, 1},
    "CholCheck": {0, 1},
    "Smoker": {0, 1},
    "Stroke": {0, 1},
    "HeartDiseaseorAttack": {0, 1},
    "PhysActivity": {0, 1},
    "Fruits": {0, 1},
    "Veggies": {0, 1},
    "HvyAlcoholConsump": {0, 1},
    "AnyHealthcare": {0, 1},
    "NoDocbcCost": {0, 1},
    "DiffWalk": {0, 1},
    "Sex": {0, 1},
    "GenHlth": set(range(1, 6)),
    "Age": set(range(1, 14)),
    "Education": set(range(1, 7)),
    "Income": set(range(1, 9)),
    "MentHlth": set(range(0, 31)),
    "PhysHlth": set(range(0, 31))
}

invalid_results = []

for column, valid_values in VALID_RANGES.items():

    if column not in df_primary.columns:
        continue

    invalid_mask = (
        ~df_primary[column].isin(valid_values)
        & df_primary[column].notna()
    )

    invalid_count = invalid_mask.sum()

    invalid_results.append({
        "Variable": column,
        "Invalid Values": invalid_count
    })

invalid_results = pd.DataFrame(invalid_results)

display(invalid_results)

,Variable,Invalid Values
0,Diabetes_012,0
1,HighBP,0
2,HighChol,0
3,CholCheck,0
4,Smoker,0
5,Stroke,0
6,HeartDiseaseorAttack,0
7,PhysActivity,0
8,Fruits,0
9,Veggies,0


In [78]:
# BMI validity check
# Values below 12 or >= 100 are treated as invalid/missing.

invalid_bmi_mask = (
    (df_primary["BMI"] < 12) |
    (df_primary["BMI"] >= 100)
) & df_primary["BMI"].notna()

print(
    f"Invalid BMI observations: "
    f"{invalid_bmi_mask.sum():,}"
)

if invalid_bmi_mask.sum() > 0:
    display(
        df_primary.loc[
            invalid_bmi_mask,
            ["BMI"]
        ].head(20)
    )

Invalid BMI observations: 0


## 8. Apply Data Cleaning

Invalid coded values are converted to missing values rather than silently deleting entire observations.

For BMI, values below 12 or at/above 100 are treated as invalid according to the source's calculated-variable coding.

The cleaning procedure is deliberately conservative: it changes values that fail validity checks without inventing replacement values.

In [79]:
df_primary_clean = clean_invalid_values(
    df_primary.copy()
)

df_binary_clean = clean_invalid_values(
    df_binary.copy()
)

df_balanced_clean = clean_invalid_values(
    df_balanced.copy()
)

print("Cleaning completed.")


No invalid values required correction.

No invalid values required correction.

No invalid values required correction.
Cleaning completed.


In [80]:
print("Primary dataset missing values after cleaning:")
display(
    df_primary_clean.isnull()
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print("\nPrimary shape after cleaning:")
print(df_primary_clean.shape)

Primary dataset missing values after cleaning:


Diabetes_012            0
HighBP                  0
HighChol                0
CholCheck               0
BMI                     0
Smoker                  0
Stroke                  0
HeartDiseaseorAttack    0
PhysActivity            0
Fruits                  0
dtype: int64


Primary shape after cleaning:
(253680, 22)


In [81]:
df_primary_clean = downcast_types(
    df_primary_clean
)

df_binary_clean = downcast_types(
    df_binary_clean
)

df_balanced_clean = downcast_types(
    df_balanced_clean
)


Memory footprint reduced from 42.58 MB to 11.13 MB (73.9% saving)

Memory footprint reduced from 42.58 MB to 11.13 MB (73.9% saving)

Memory footprint reduced from 11.87 MB to 3.10 MB (73.9% saving)


## 10. Feature Engineering

Three project-defined analytical features are created:

### `Is_Obese`
Binary indicator based on BMI ≥ 30.

### `Metabolic_Factor_Count`
Counts the presence of three indicators:

- High blood pressure
- High cholesterol
- Obesity

This is a project-defined analytical count, **not an established clinical score**.

### `Healthy_Habits`
A 0–4 exploratory index based on:

- Physical activity
- Fruit consumption
- Vegetable consumption
- Non-smoking

The components are equally weighted for this project and should not be interpreted as a validated clinical index.

### `Combined_Health_Burden`
The sum of `MentHlth` and `PhysHlth`.

Because the two measures may overlap, this is **not interpreted as a count of unique unhealthy days**.

In [82]:
df_primary_clean = engineer_features(
    df_primary_clean
)

df_binary_clean = engineer_features(
    df_binary_clean
)

df_balanced_clean = engineer_features(
    df_balanced_clean
)

print("Feature engineering completed.")

Feature engineering completed.


In [83]:
engineered_columns = [
    "Is_Obese",
    "Metabolic_Factor_Count",
    "Non_Smoker",
    "Healthy_Habits",
    "Combined_Health_Burden"
]

display(
    df_primary_clean[
        engineered_columns
    ].head(10)
)

,Is_Obese,Metabolic_Factor_Count,Non_Smoker,Healthy_Habits,Combined_Health_Burden
0,1,3,0,1,33
1,0,0,0,1,0
2,0,2,1,2,60
3,0,1,1,4,0
4,0,2,1,4,3
5,0,2,0,3,2
6,1,2,0,0,14
7,0,2,0,2,0
8,1,3,0,2,60
9,0,0,1,2,0


## 11. Final Data Quality Check

In [84]:
final_audit = pd.DataFrame({
    "Dataset": [
        "Primary",
        "Binary",
        "Balanced"
    ],
    "Rows": [
        len(df_primary_clean),
        len(df_binary_clean),
        len(df_balanced_clean)
    ],
    "Columns": [
        df_primary_clean.shape[1],
        df_binary_clean.shape[1],
        df_balanced_clean.shape[1]
    ],
    "Missing Values": [
        df_primary_clean.isnull().sum().sum(),
        df_binary_clean.isnull().sum().sum(),
        df_balanced_clean.isnull().sum().sum()
    ],
    "Duplicate Rows": [
        df_primary_clean.duplicated().sum(),
        df_binary_clean.duplicated().sum(),
        df_balanced_clean.duplicated().sum()
    ]
})

display(final_audit)

,Dataset,Rows,Columns,Missing Values,Duplicate Rows
0,Primary,253680,27,0,23899
1,Binary,253680,27,0,24206
2,Balanced,70692,27,0,1635


## 12. Final Feature Inventory

The cleaned primary dataset now contains the original BRFSS variables plus project-defined analytical features.

These processed datasets will be used in the next stages:

- Exploratory analysis
- Statistical inference
- Multivariable modelling
- Predictive modelling
- Final data-story visualizations

In [85]:
feature_inventory = pd.DataFrame({
    "Variable": df_primary_clean.columns,
    "Data Type": [
        str(dtype)
        for dtype in df_primary_clean.dtypes
    ]
})

display(feature_inventory)

,Variable,Data Type
0,Diabetes_012,Int8
1,HighBP,Int8
2,HighChol,Int8
3,CholCheck,Int8
4,BMI,float32
5,Smoker,Int8
6,Stroke,Int8
7,HeartDiseaseorAttack,Int8
8,PhysActivity,Int8
9,Fruits,Int8


## 13. Data Dictionary

In [86]:
data_dictionary = pd.DataFrame({

    "Variable": [
        "Diabetes_012",
        "HighBP",
        "HighChol",
        "CholCheck",
        "BMI",
        "Smoker",
        "Stroke",
        "HeartDiseaseorAttack",
        "PhysActivity",
        "Fruits",
        "Veggies",
        "HvyAlcoholConsump",
        "AnyHealthcare",
        "NoDocbcCost",
        "GenHlth",
        "MentHlth",
        "PhysHlth",
        "DiffWalk",
        "Sex",
        "Age",
        "Education",
        "Income",
        "Is_Obese",
        "Metabolic_Factor_Count",
        "Non_Smoker",
        "Healthy_Habits",
        "Combined_Health_Burden"
    ],

    "Description": [
        "Diabetes status: 0 = No Diabetes, 1 = Prediabetes, 2 = Diabetes",
        "High blood pressure",
        "High cholesterol",
        "Cholesterol check within the previous 5 years",
        "Body Mass Index",
        "Current smoking indicator",
        "History of stroke",
        "History of coronary heart disease or myocardial infarction",
        "Physical activity indicator",
        "Fruit consumption indicator",
        "Vegetable consumption indicator",
        "Heavy alcohol consumption indicator",
        "Health-care coverage indicator",
        "Could not see a doctor because of cost",
        "Self-reported general health",
        "Number of mentally unhealthy days in the past 30 days",
        "Number of physically unhealthy days in the past 30 days",
        "Difficulty walking or climbing stairs",
        "Sex category",
        "Age category",
        "Education category",
        "Income category",
        "Project-derived obesity indicator: BMI >= 30",
        "Project-defined count of high BP, high cholesterol and obesity",
        "Project-derived non-smoking indicator",
        "Project-defined 0–4 healthy-habits index",
        "Project-defined sum of MentHlth and PhysHlth"
    ]
})

display(data_dictionary)

,Variable,Description
0,Diabetes_012,"Diabetes status: 0 = No Diabetes, 1 = Prediabe..."
1,HighBP,High blood pressure
2,HighChol,High cholesterol
3,CholCheck,Cholesterol check within the previous 5 years
4,BMI,Body Mass Index
5,Smoker,Current smoking indicator
6,Stroke,History of stroke
7,HeartDiseaseorAttack,History of coronary heart disease or myocardia...
8,PhysActivity,Physical activity indicator
9,Fruits,Fruit consumption indicator


## 14. Save Processed Datasets

The cleaned datasets are saved separately so that every downstream notebook can work from reproducible processed files rather than repeatedly modifying the raw data.

The raw datasets are preserved unchanged.

In [87]:
primary_output = os.path.join(
    PROCESSED_DIR,
    "diabetes_progression_cleaned.csv"
)

binary_output = os.path.join(
    PROCESSED_DIR,
    "diabetes_full_cleaned.csv"
)

balanced_output = os.path.join(
    PROCESSED_DIR,
    "diabetes_balanced_cleaned.csv"
)

df_primary_clean.to_csv(
    primary_output,
    index=False
)

df_binary_clean.to_csv(
    binary_output,
    index=False
)

df_balanced_clean.to_csv(
    balanced_output,
    index=False
)

print("Processed datasets saved successfully:\n")

print(primary_output)
print(binary_output)
print(balanced_output)

Processed datasets saved successfully:

../data/processed\diabetes_progression_cleaned.csv
../data/processed\diabetes_full_cleaned.csv
../data/processed\diabetes_balanced_cleaned.csv


# Phase 1 Complete

The datasets have now been:

- audited for structure and quality
- checked for invalid coded values
- checked for BMI validity
- cleaned using source-compatible rules
- memory optimized
- augmented with project-defined analytical features
- documented with a data dictionary
- saved as reproducible processed datasets

### Primary dataset for the project

`diabetes_progression_cleaned.csv`

### Primary outcome

`Diabetes_012`

### Outcome interpretation

`0 = No Diabetes`  
`1 = Prediabetes`  
`2 = Diabetes`

The next phase will use this cleaned primary dataset for exploratory analysis and visual storytelling.